In [1]:
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 138.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 190.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]


In [2]:
!pip install numpy

In [ ]:
import pandas as pd
import numpy as np
import os

In [8]:
# 1. Definimos los nombres de las 26 columnas originales de acuerdo con las especificaciones de la NASA
column_names = ['unit_number', 'cycle', 'setting_1', 'setting_2', 'setting_3'] + [f'sensor_{i}' for i in range(1, 22)]

# 2. Especificamos el nombre de la carpeta donde están guardados tus archivos de texto
data_folder = 'raw_data'

# 3. Cargamos los 4 archivos usando os.path.join para que busque correctamente dentro de la carpeta 'raw_data'
# El parámetro sep=r'\s+' se encarga de limpiar cualquier espacio extra o tabulación en el archivo de texto
df1 = pd.read_csv(os.path.join(data_folder, 'train_FD001.txt'), sep=r'\s+', header=None, names=column_names)
df2 = pd.read_csv(os.path.join(data_folder, 'train_FD002.txt'), sep=r'\s+', header=None, names=column_names)
df3 = pd.read_csv(os.path.join(data_folder, 'train_FD003.txt'), sep=r'\s+', header=None, names=column_names)
df4 = pd.read_csv(os.path.join(data_folder, 'train_FD004.txt'), sep=r'\s+', header=None, names=column_names)

# 4. Añadimos la columna identificadora del dataset original a cada DataFrame separado
df1['dataset'] = 'FD001'
df2['dataset'] = 'FD002'
df3['dataset'] = 'FD003'
df4['dataset'] = 'FD004'

In [12]:
df1_table = pd.DataFrame(df1)

In [13]:
df1_table

,unit_number,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,dataset
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,FD001
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,FD001
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,FD001
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,FD001
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,FD001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20626,100,196,-0.0004,-0.0003,100.0,518.67,643.49,1597.98,1428.63,14.62,...,2388.26,8137.60,8.4956,0.03,397,2388,100.0,38.49,22.9735,FD001
20627,100,197,-0.0016,-0.0005,100.0,518.67,643.54,1604.50,1433.58,14.62,...,2388.22,8136.50,8.5139,0.03,395,2388,100.0,38.30,23.1594,FD001
20628,100,198,0.0004,0.0000,100.0,518.67,643.42,1602.46,1428.18,14.62,...,2388.24,8141.05,8.5646,0.03,398,2388,100.0,38.44,22.9333,FD001
20629,100,199,-0.0011,0.0003,100.0,518.67,643.23,1605.26,1426.53,14.62,...,2388.23,8139.29,8.5389,0.03,395,2388,100.0,38.29,23.0640,FD001


In [ ]:
df2_table = pd.DataFrame(df1)

In [ ]:

# 4. Creamos una función reutilizable para calcular la variable objetivo Y (RUL) y estructurar los identificadores
def procesar_dataset_inicial(df):
    # Agrupamos los datos por cada motor ('unit_number') y extraemos el valor más alto de la columna 'cycle'
    # Como los motores de entrenamiento se registran hasta que fallan, el ciclo máximo representa el momento exacto de la falla
    max_cycles = df.groupby('unit_number')['cycle'].max().reset_index()

    # Renombramos la columna resultante a 'max_cycle' para que no choque con la columna 'cycle' original al hacer la fusión
    max_cycles.columns = ['unit_number', 'max_cycle']

    # Cruzamos el DataFrame original con nuestra tabla de ciclos máximos usando la columna 'unit_number' como llave de unión
    df = pd.merge(df, max_cycles, on='unit_number')

    # Calculamos la variable Y (RUL) restando algebraicamente: Ciclo Máximo de Falla - Ciclo de la Fila Actual
    df['RUL'] = df['max_cycle'] - df['cycle']

    # Aplicamos Piecewise RUL (RUL Segmentado): si el resultado es mayor a 125, lo limita automáticamente a 125
    # Esto estabiliza el entrenamiento ya que el desgaste real no empieza desde el ciclo 1 y homogeniza los 4 datasets
    df['RUL'] = df['RUL'].clip(upper=125)

    # Eliminamos la columna auxiliar 'max_cycle' utilizando inplace=True para liberar memoria RAM de la máquina virtual
    df.drop(columns=['max_cycle'], inplace=True)

    # Creamos un código de identificación único para cada motor combinando el nombre de su dataset y su número de unidad
    # Esto evita que el Motor 1 de FD001 se mezcle o se confunda con el Motor 1 de FD002 al unificar las tablas
    df['unique_motor_id'] = df['dataset'] + "_" + df['unit_number'].astype(str)

    # Retornamos el DataFrame completamente procesado y con su columna objetivo Y ('RUL') calculada
    return df

# 5. Ejecutamos la función de procesamiento de forma secuencial en cada uno de los 4 subconjuntos de datos
df1 = procesar_dataset_inicial(df1)
df2 = procesar_dataset_inicial(df2)
df3 = procesar_dataset_inicial(df3)
df4 = procesar_dataset_inicial(df4)

# 6. Concatenamos verticalmente los 4 DataFrames individuales en una sola gran matriz unificada de entrenamiento
# El parámetro ignore_index=True reestructura los índices numéricos de las filas desde 0 hasta el total general de filas
df_global = pd.concat([df1, df2, df3, df4], ignore_index=True)

# 7. Imprimimos un resumen en la consola para validar que los datos se hayan unificado correctamente
print(f"¡Paso 1 Completado exitosamente!")
print(f"Dimensiones de la matriz global unificada: {df_global.shape[0]} filas y {df_global.shape[1]} columnas.")
print(f"Cantidad de motores únicos en la flota global: {df_global['unique_motor_id'].nunique()}")